# Tiền xử lý dữ liệu Abalone

Sổ tay này thực hiện các bước tiền xử lý dữ liệu cho bộ dữ liệu Abalone dựa trên các phát hiện từ phần khám phá dữ liệu (EDA). Nội dung gồm: kiểm tra dữ liệu thiếu, mã hóa biến phân loại, xử lý ngoại lệ, chuẩn hóa dữ liệu và chia tập huấn luyện - kiểm tra.

## 1. Import thư viện cần thiết

In [23]:
import io
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Nạp và kiểm tra dữ liệu

In [24]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'abalone.csv'
INTERIM_DATA_PATH = PROJECT_ROOT / 'data' / 'interim'
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'

# Tạo thư mục nếu chưa tồn tại
INTERIM_DATA_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

columns = [
    'Sex',
    'Length',
    'Diameter',
    'Height',
    'WholeWeight',
    'ShuckedWeight',
    'VisceraWeight',
    'ShellWeight',
    'Rings',
]

df = pd.read_csv(DATA_PATH, header=None, names=columns)
print(f'Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột')
print('\nKiểu dữ liệu của từng cột:')
display(df.dtypes.to_frame('dtype'))
display(df.head())

Kích thước dữ liệu: 4177 dòng, 9 cột

Kiểu dữ liệu của từng cột:


,dtype
Sex,object
Length,float64
Diameter,float64
Height,float64
WholeWeight,float64
ShuckedWeight,float64
VisceraWeight,float64
ShellWeight,float64
Rings,int64


,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


## 3. Kiểm tra giá trị thiếu và dữ liệu trùng lặp

Từ EDA, chúng ta biết rằng không có giá trị thiếu, nhưng vẫn kiểm tra lại để chứng minh chất lượng dữ liệu.

In [ ]:
missing_summary = df.isna().sum().to_frame('missing_count')
missing_summary['missing_ratio'] = (missing_summary['missing_count'] / len(df)).round(4)

duplicate_count = df.duplicated().sum()

display(missing_summary)
print(f'\nSố dòng trùng lặp: {duplicate_count}')
print('\nKết luận: Không có giá trị thiếu, có thể bỏ qua bước xử lý dữ liệu khuyết.')

,missing_count,missing_ratio
Sex,0,0.0
Length,0,0.0
Diameter,0,0.0
Height,0,0.0
WholeWeight,0,0.0
ShuckedWeight,0,0.0
VisceraWeight,0,0.0
ShellWeight,0,0.0
Rings,0,0.0



Số dòng trùng lặp: 0

Kết luận: Không có giá trị thiếu, có thể bỏ qua bước xử lý missing value.


## 4. Mã hóa biến phân loại `Sex`

Sử dụng mã hóa one-hot để chuyển đổi biến `Sex` sang dạng số mà không áp đặt thứ tự giả.

In [ ]:
# Tách biến phân loại và biến số
categorical_cols = ['Sex']
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Loại bỏ 'Rings' khỏi danh sách biến số vì đây là biến mục tiêu
target_col = 'Rings'
numeric_cols = [col for col in numeric_cols if col != target_col]

print(f'Biến phân loại: {categorical_cols}')
print(f'Biến số: {numeric_cols}')
print(f'Biến mục tiêu: {target_col}')

# Áp dụng mã hóa one-hot cho biến Sex
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)
print(f'\nKích thước sau mã hóa: {df_encoded.shape}')
display(df_encoded.head())

Biến phân loại: ['Sex']
Biến số: ['Length', 'Diameter', 'Height', 'WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight']
Biến mục tiêu: Rings

Hình dạng sau mã hóa: (4177, 11)


,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings,Sex_F,Sex_I,Sex_M
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,False,False,True
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,False,False,True
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,True,False,False
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,False,False,True
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,False,True,False


## 5. Xử lý ngoại lệ

Dựa trên EDA, áp dụng ba chiến lược xử lý ngoại lệ:
- `baseline`: Giữ nguyên dữ liệu gốc
- `iqr_filtered`: Lọc ngoại lệ theo phương pháp IQR
- `log_transformed`: Biến đổi log cho các biến khối lượng

In [ ]:
# Chiến lược 1: baseline - giữ nguyên dữ liệu
df_baseline = df_encoded.copy()

# Chiến lược 2: lọc ngoại lệ theo IQR
def remove_outliers_iqr(data, numeric_cols):
    """
    Loại bỏ các dòng chứa ngoại lệ theo phương pháp IQR.
    """
    df_clean = data.copy()
    initial_len = len(df_clean)

    for col in numeric_cols:
        q1 = df_clean[col].quantile(0.25)
        q3 = df_clean[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]

    removed_count = initial_len - len(df_clean)
    print(f'Số dòng bị loại bởi lọc IQR: {removed_count} ({100*removed_count/initial_len:.2f}%)')
    return df_clean

df_iqr = remove_outliers_iqr(df_encoded, numeric_cols)

# Chiến lược 3: biến đổi log cho các biến khối lượng
weight_cols = ['WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight']

df_log = df_encoded.copy()
for col in weight_cols:
    df_log[f'{col}_log'] = np.log1p(df_log[col])  # log1p để tránh log(0)

print(f'\nKích thước dữ liệu sau biến đổi log: {df_log.shape}')
print(f'Các cột mới: {[col for col in df_log.columns if "_log" in col]}')
display(df_log.head())

Số dòng bị loại bỏ bởi IQR filtering: 164 (3.93%)

Hình dạng dữ liệu sau log transformation: (4177, 15)
Các cột mới: ['WholeWeight_log', 'ShuckedWeight_log', 'VisceraWeight_log', 'ShellWeight_log']


,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings,Sex_F,Sex_I,Sex_M,WholeWeight_log,ShuckedWeight_log,VisceraWeight_log,ShellWeight_log
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,False,False,True,0.414755,0.202533,0.096219,0.139762
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,False,False,True,0.203349,0.094856,0.047361,0.067659
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,True,False,False,0.517006,0.228330,0.132343,0.190620
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,False,False,True,0.416075,0.195156,0.107957,0.144100
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,False,True,False,0.186480,0.085719,0.038740,0.053541


## 6. Chia dữ liệu thành tập huấn luyện - kiểm tra

Chia dữ liệu theo tỷ lệ 70/30 với `random_state=42` để đảm bảo khả năng tái lập kết quả.

In [ ]:
RANDOM_STATE = 42
TRAIN_SIZE = 0.7

def split_data(df, target_col='Rings', random_state=RANDOM_STATE, train_size=TRAIN_SIZE):
    """
    Chia dữ liệu thành tập huấn luyện và kiểm tra.
    """
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=1-train_size, random_state=random_state
    )

    print(f'Kích thước tập huấn luyện: {X_train.shape}')
    print(f'Kích thước tập kiểm tra: {X_test.shape}')
    print(f'Tỷ lệ huấn luyện/kiểm tra: {train_size}/{1-train_size}')

    return X_train, X_test, y_train, y_test

# Chia dữ liệu cho cả ba chiến lược
datasets = {
    'baseline': df_baseline,
    'iqr_filtered': df_iqr,
    'log_transformed': df_log
}

split_results = {}
for name, df_strategy in datasets.items():
    print(f'\n=== {name.upper()} ===')
    X_train, X_test, y_train, y_test = split_data(df_strategy)
    split_results[name] = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }


=== BASELINE ===
Kích thước train set: (2923, 10)
Kích thước test set: (1254, 10)
Tỷ lệ train/test: 0.7/0.30000000000000004

=== IQR_FILTERED ===
Kích thước train set: (2809, 10)
Kích thước test set: (1204, 10)
Tỷ lệ train/test: 0.7/0.30000000000000004

=== LOG_TRANSFORMED ===
Kích thước train set: (2923, 14)
Kích thước test set: (1254, 14)
Tỷ lệ train/test: 0.7/0.30000000000000004


## 7. Chuẩn hóa dữ liệu

Sử dụng `StandardScaler` và `RobustScaler` cho các biến số. `StandardScaler` phù hợp khi cần chuẩn hóa theo trung bình và độ lệch chuẩn, còn `RobustScaler` ổn định hơn khi dữ liệu có ngoại lệ.

In [ ]:
# Xác định các biến cần chuẩn hóa (loại trừ biến one-hot và biến log)
sex_cols = [col for col in split_results['baseline']['X_train'].columns if col.startswith('Sex_')]
base_numeric_cols = numeric_cols.copy()

print(f'Các biến Sex (one-hot): {sex_cols}')
print(f'Các biến số cần chuẩn hóa: {base_numeric_cols}')

# Hàm chuẩn hóa dữ liệu
def scale_data(X_train, X_test, numeric_cols, scaler_type='standard'):
    """
    Chuẩn hóa dữ liệu bằng StandardScaler hoặc RobustScaler.
    """
    # Chỉ chuẩn hóa các biến số gốc
    cols_to_scale = [col for col in numeric_cols if col in X_train.columns]

    if scaler_type == 'standard':
        scaler = StandardScaler()
    elif scaler_type == 'robust':
        scaler = RobustScaler()
    else:
        raise ValueError(f'Kiểu scaler không hợp lệ: {scaler_type}')

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
    X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

    return X_train_scaled, X_test_scaled, scaler

# Áp dụng chuẩn hóa cho tất cả chiến lược
scaled_results = {}
for strategy_name, data in split_results.items():
    print(f'\n=== Chuẩn hóa cho chiến lược: {strategy_name} ===')

    # Xác định các biến số cần chuẩn hóa theo từng chiến lược
    if strategy_name == 'log_transformed':
        # Bao gồm cả các biến đã biến đổi log
        cols_to_scale = [
            col for col in base_numeric_cols + [f'{col}_log' for col in weight_cols]
            if col in data['X_train'].columns
        ]
    else:
        cols_to_scale = [col for col in base_numeric_cols if col in data['X_train'].columns]

    # StandardScaler
    X_train_std, X_test_std, scaler_std = scale_data(
        data['X_train'], data['X_test'], cols_to_scale, scaler_type='standard'
    )

    # RobustScaler
    X_train_robust, X_test_robust, scaler_robust = scale_data(
        data['X_train'], data['X_test'], cols_to_scale, scaler_type='robust'
    )

    scaled_results[strategy_name] = {
        'standard': {
            'X_train': X_train_std,
            'X_test': X_test_std,
            'scaler': scaler_std
        },
        'robust': {
            'X_train': X_train_robust,
            'X_test': X_test_robust,
            'scaler': scaler_robust
        },
        'y_train': data['y_train'],
        'y_test': data['y_test']
    }

print('\n✓ Chuẩn hóa dữ liệu hoàn tất')

Biến Sex (one-hot encoded): ['Sex_F', 'Sex_I', 'Sex_M']
Biến số cần scale: ['Length', 'Diameter', 'Height', 'WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight']

=== Scaling for baseline ===

=== Scaling for iqr_filtered ===

=== Scaling for log_transformed ===

✓ Chuẩn hóa dữ liệu hoàn tất


## 8. Tạo đặc trưng (mức cơ bản)

Tạo các đặc trưng mới dựa trên kết quả phân tích dữ liệu khám phá, chủ yếu là các tỷ lệ giữa khối lượng và kích thước.

In [ ]:
def create_features(df, target_col='Rings'):
    """
    Tạo các đặc trưng mới dựa trên dữ liệu hiện có.
    """
    df_new = df.copy()
    # Tỷ lệ các thành phần khối lượng so với khối lượng tổng
    df_new['ShuckedWeight_to_WholeWeight'] = df_new['ShuckedWeight'] / (df_new['WholeWeight'] + 1e-8)
    df_new['VisceraWeight_to_WholeWeight'] = df_new['VisceraWeight'] / (df_new['WholeWeight'] + 1e-8)
    df_new['ShellWeight_to_WholeWeight'] = df_new['ShellWeight'] / (df_new['WholeWeight'] + 1e-8)

    # Thể tích xấp xỉ từ các kích thước
    df_new['Volume'] = df_new['Length'] * df_new['Diameter'] * df_new['Height']

    # Mật độ xấp xỉ
    df_new['Density'] = df_new['WholeWeight'] / (df_new['Volume'] + 1e-8)
    return df_new
# Áp dụng tạo đặc trưng cho chiến lược nền
print('TẠO ĐẶC TRƯNG CHO BỘ DỮ LIỆU NỀN')
df_baseline_fe = create_features(df_baseline)
print(f'Kích thước sau khi tạo đặc trưng: {df_baseline_fe.shape}')
print(f'Các đặc trưng mới: {[col for col in df_baseline_fe.columns if col not in df_baseline.columns]}')

# Chia dữ liệu mới
X_train_fe, X_test_fe, y_train_fe, y_test_fe = split_data(df_baseline_fe)

# Chuẩn hóa dữ liệu mới
fe_numeric_cols = [col for col in base_numeric_cols if col in X_train_fe.columns] + \
                  [col for col in X_train_fe.columns if any(x in col for x in ['_to_', 'Volume', 'Density'])]

X_train_fe_scaled, X_test_fe_scaled, scaler_fe = scale_data(
    X_train_fe, X_test_fe, fe_numeric_cols, scaler_type='standard'
)
print('\nTạo đặc trưng và chuẩn hóa dữ liệu hoàn tất')

=== Creating Features for Baseline Dataset ===
Hình dạng sau feature engineering: (4177, 16)
Các đặc trưng mới: ['ShuckedWeight_to_WholeWeight', 'VisceraWeight_to_WholeWeight', 'ShellWeight_to_WholeWeight', 'Volume', 'Density']
Kích thước train set: (2923, 15)
Kích thước test set: (1254, 15)
Tỷ lệ train/test: 0.7/0.30000000000000004

✓ Feature engineering và scaling hoàn tất


## 9. Lưu dữ liệu đã xử lý

In [ ]:
import os
import pickle

# Lưu các bộ dữ liệu đã tiền xử lý và chia train-test
def save_preprocessed_data(split_results, scaled_results, strategy_name, save_path):
    """
    Lưu dữ liệu đã tiền xử lý và scaler
    """
    os.makedirs(save_path, exist_ok=True)
    # Lưu dữ liệu đã chia train-test
    with open(save_path / f"{strategy_name}_split.pkl", "wb") as f:
        pickle.dump(split_results[strategy_name], f)
    # Lưu dữ liệu đã chuẩn hóa và scaler
    with open(save_path / f"{strategy_name}_scaled.pkl", "wb") as f:
        pickle.dump(scaled_results[strategy_name], f)
    print(f" Đã lưu dữ liệu: {strategy_name}")
# Lưu dữ liệu cho thư mục interim
print(" ĐANG LƯU VÀO THƯ MỤC INTERIM ")
for strategy_name in ["baseline", "iqr_filtered", "log_transformed"]:
    save_preprocessed_data(split_results, scaled_results, strategy_name, INTERIM_DATA_PATH)
# Lưu dữ liệu feature engineering cho thư mục processed
print("\nĐANG LƯU DỮ LIỆU ĐẶC TRƯNG MỞ RỘNG VÀO THƯ MỤC PROCESSED")
fe_processed_data = {
    "X_train": X_train_fe_scaled,
    "X_test": X_test_fe_scaled,
    "y_train": y_train_fe,
    "y_test": y_test_fe,
    "scaler": scaler_fe,
}
with open(PROCESSED_DATA_PATH / "baseline_with_features_scaled.pkl", "wb") as f:
    pickle.dump(fe_processed_data, f)
# Lưu các phiên bản chưa chuẩn hóa
with open(PROCESSED_DATA_PATH / "baseline_with_features_unscaled.pkl", "wb") as f:
    pickle.dump(
        {
            "X_train": X_train_fe,
            "X_test": X_test_fe,
            "y_train": y_train_fe,
            "y_test": y_test_fe,
        },
        f,
    )
# Lưu các bộ dữ liệu gốc đã mã hóa
print("=== ĐANG LƯU CÁC BỘ DỮ LIỆU ĐÃ MÃ HÓA ===")
for strategy_name in ["baseline", "iqr_filtered", "log_transformed"]:
    with open(PROCESSED_DATA_PATH / f"{strategy_name}_full_encoded.pkl", "wb") as f:
        pickle.dump(datasets[strategy_name], f)

# Tạo tệp tóm tắt tiền xử lý
summary_text = f"""
TÓM TẮT TIỀN XỬ LÝ DỮ LIỆU 

Bộ dữ liệu nền (đã mã hóa):
- Kích thước: {df_baseline.shape}
- Các cột: {df_baseline.columns.tolist()}

Bộ dữ liệu sau lọc IQR:
- Số dòng ban đầu: {df_encoded.shape[0]}
- Số dòng sau lọc: {df_iqr.shape[0]}
- Số dòng bị loại: {df_encoded.shape[0] - df_iqr.shape[0]} ({100*(df_encoded.shape[0] - df_iqr.shape[0])/df_encoded.shape[0]:.2f}%)

Bộ dữ liệu sau biến đổi log:
- Các cột trọng lượng được biến đổi: {weight_cols}
- Các cột log mới: {[col for col in df_log.columns if '_log' in col]}

Chia train-test:
- random_state: {RANDOM_STATE}
- Tỷ lệ train: {TRAIN_SIZE}
- Tỷ lệ test: {1-TRAIN_SIZE}

Phương pháp chuẩn hóa đã áp dụng:
- StandardScaler (trung bình bằng 0, độ lệch chuẩn bằng 1)
- RobustScaler (dựa trên trung vị và IQR, ít nhạy với ngoại lệ)

Đặc trưng mở rộng:
- Các đặc trưng mới: ShuckedWeight_to_WholeWeight, VisceraWeight_to_WholeWeight, ShellWeight_to_WholeWeight, Volume, Density

Các bộ dữ liệu đã lưu:
- interim/: dữ liệu chia train-test và dữ liệu đã chuẩn hóa theo từng chiến lược
- processed/: dữ liệu hoàn thiện, bao gồm đặc trưng mở rộng

Bước tiếp theo:
Dùng các bộ dữ liệu đã chuẩn hóa trong notebook 03_huan_luyen_mo_hinh.ipynb để huấn luyện và so sánh mô hình.
"""

with open(PROCESSED_DATA_PATH / "PREPROCESSING_SUMMARY.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)
print(summary_text)

=== Saving to interim folder ===
✓ Saved baseline data
✓ Saved iqr_filtered data
✓ Saved log_transformed data

=== Saving Feature Engineering Data to Processed folder ===
=== Saving Original Encoded Datasets ===

=== DATA PREPROCESSING SUMMARY ===

Dataset Baseline (Encoded):
- Shape: (4177, 11)
- Columns: ['Length', 'Diameter', 'Height', 'WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight', 'Rings', 'Sex_F', 'Sex_I', 'Sex_M']

Dataset IQR Filtered:
- Original shape: 4177
- Filtered shape: 4013
- Rows removed: 164 (3.93%)

Dataset Log Transformed:
- Weight columns transformed: ['WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight']
- New columns: ['WholeWeight_log', 'ShuckedWeight_log', 'VisceraWeight_log', 'ShellWeight_log']

Train-Test Split:
- Random state: 42
- Train ratio: 0.7
- Test ratio: 0.30000000000000004

Scaling Methods Applied:
- StandardScaler (zero mean, unit variance)
- RobustScaler (using median and IQR, robust to outliers)

Feature Engineering:
- Ne

## 10. Xác minh dữ liệu đã xử lý

Kiểm tra kích thước, kiểu dữ liệu, và số liệu thống kê cơ bản của dữ liệu đã xử lý.

In [ ]:
# Kiểm tra dữ liệu huấn luyện sau khi chuẩn hóa
print('XÁC MINH DỮ LIỆU ĐÃ CHUẨN HÓA\n')

print('Chiến lược nền (StandardScaler):')
print(f'Kích thước X_train: {scaled_results["baseline"]["standard"]["X_train"].shape}')
print(f'Kích thước X_test: {scaled_results["baseline"]["standard"]["X_test"].shape}')
print(f'Kích thước y_train: {scaled_results["baseline"]["y_train"].shape}')
print(f'Kích thước y_test: {scaled_results["baseline"]["y_test"].shape}')
print('\nThống kê train của chiến lược nền (một số cột đầu):')
display(scaled_results["baseline"]["standard"]["X_train"].iloc[:, :5].describe())

print('\n\nChiến lược nền (RobustScaler):')
print(f'Kích thước X_train: {scaled_results["baseline"]["robust"]["X_train"].shape}')
print(f'Kích thước X_test: {scaled_results["baseline"]["robust"]["X_test"].shape}')

print('\n\nChiến lược lọc IQR:')
print(f'Kích thước X_train: {scaled_results["iqr_filtered"]["standard"]["X_train"].shape}')
print(f'Kích thước X_test: {scaled_results["iqr_filtered"]["standard"]["X_test"].shape}')

print('\n\nChiến lược biến đổi log:')
print(f'Kích thước X_train: {scaled_results["log_transformed"]["standard"]["X_train"].shape}')
print(f'Kích thước X_test: {scaled_results["log_transformed"]["standard"]["X_test"].shape}')
print('\nCác cột log mới:')
log_cols = [col for col in scaled_results["log_transformed"]["standard"]["X_train"].columns if '_log' in col]
print(f'Danh sách cột log: {log_cols}')

print('\n\nDữ liệu có đặc trưng mở rộng từ chiến lược nền:')
print(f'Kích thước X_train: {X_train_fe_scaled.shape}')
print(f'Kích thước X_test: {X_test_fe_scaled.shape}')
fe_cols = [col for col in X_train_fe_scaled.columns if any(x in col for x in ['_to_', 'Volume', 'Density'])]
print(f'Các cột đặc trưng mở rộng: {fe_cols}')

print('\n Tiền xử lý dữ liệu thành công! Sẵn sàng cho bước huấn luyện mô hình.')

=== XÁC MINH DỮ LIỆU ĐÃ CHUẨN HÓA ===

Chiến lược nền (StandardScaler):
Kích thước X_train: (2923, 10)
Kích thước X_test: (1254, 10)
Kích thước y_train: (2923,)
Kích thước y_test: (1254,)

Thống kê train của chiến lược nền (một số cột đầu):


,Length,Diameter,Height,WholeWeight,ShuckedWeight
count,2.923000e+03,2.923000e+03,2.923000e+03,2.923000e+03,2.923000e+03
mean,4.740193e-16,4.290482e-16,4.278328e-16,1.093891e-16,-2.260707e-16
std,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00
min,-3.768867e+00,-3.581951e+00,-3.298727e+00,-1.695805e+00,-1.619665e+00
25%,-6.360994e-01,-6.014650e-01,-5.915393e-01,-7.979302e-01,-7.874196e-01
50%,1.575351e-01,1.562855e-01,1.146837e-01,-5.656070e-02,-1.028309e-01
75%,7.423184e-01,7.624859e-01,5.854990e-01,6.583858e-01,6.499930e-01
max,2.413128e+00,2.429537e+00,2.330234e+01,4.043076e+00,5.033822e+00




Chiến lược nền (RobustScaler):
Kích thước X_train: (2923, 10)
Kích thước X_test: (1254, 10)


Chiến lược lọc IQR:
Kích thước X_train: (2809, 10)
Kích thước X_test: (1204, 10)


Chiến lược biến đổi log:
Kích thước X_train: (2923, 14)
Kích thước X_test: (1254, 14)

Các cột log mới:
Danh sách cột log: ['WholeWeight_log', 'ShuckedWeight_log', 'VisceraWeight_log', 'ShellWeight_log']


Dữ liệu có đặc trưng mở rộng từ chiến lược nền:
Kích thước X_train: (2923, 15)
Kích thước X_test: (1254, 15)
Các cột đặc trưng mở rộng: ['ShuckedWeight_to_WholeWeight', 'VisceraWeight_to_WholeWeight', 'ShellWeight_to_WholeWeight', 'Volume', 'Density']

✓ Tiền xử lý dữ liệu thành công! Sẵn sàng cho bước huấn luyện mô hình.


## 11. Kết luận

Sau bước tiền xử lý, dữ liệu đã sẵn sàng cho giai đoạn huấn luyện mô hình với các điểm chính:

- Dữ liệu gốc không có giá trị khuyết và không có bản ghi trùng lặp đáng kể, nên không cần bù khuyết dữ liệu.
- Biến phân loại `Sex` đã được mã hóa đầy đủ bằng one-hot, phù hợp cho các mô hình hồi quy.
- Ba chiến lược xử lý ngoại lệ đã được chuẩn bị song song (`baseline`, `iqr_filtered`, `log_transformed`) để so sánh công bằng.
- Dữ liệu đã được chia train-test theo `random_state=42`, giúp tái lập kết quả giữa các lần chạy.
- Hai phương pháp chuẩn hóa (`StandardScaler`, `RobustScaler`) đã được áp dụng và lưu lại để thử nghiệm với nhiều mô hình.
- Bộ đặc trưng mở rộng đã được tạo và lưu riêng để kiểm định mức cải thiện hiệu năng.

Từ các kết quả trên, notebook tiếp theo sẽ tập trung vào:

1. Huấn luyện nhiều mô hình trên từng chiến lược dữ liệu.
2. So sánh bằng các chỉ số `MAE`, `RMSE`, `R2`.
3. Chọn cấu hình tốt nhất để phục vụ đánh giá chi tiết ở bước kế tiếp.